# Unidad 4 — Estadística Descriptiva

En esta clase vamos a explorar cómo **resumir y visualizar** un conjunto de datos usando estadística descriptiva.

Continuamos trabajando con el dataset de airbnb_jr

---
**Contenidos:**
1. Terminología básica
2. El dataset: primeros pasos
3. Medidas de tendencia central
4. Visualización: histogramas
5. Medidas de dispersión
6. Cuartiles, RIC y detección de outliers
7. Boxplot
8. Ejercicio integrador

## 1. Terminología básica

Antes de arrancar, definamos los conceptos clave:

| Concepto | Definición | Ejemplo en nuestro dataset |
|---|---|---|
| **Observación** | Unidad sobre la que se miden los datos | Una propiedad en venta |
| **Población** | Conjunto de todas las unidades de estudio | Todas las propiedades de CABA |
| **Muestra** | Subconjunto representativo de la población | Las 200 propiedades de nuestro dataset |
| **Variable** | Característica que se mide en cada observación | Precio, superficie, barrio |
| **Variable cuantitativa** | Toma valores numéricos (se puede calcular la media) | Precio, superficie |
| **Variable cualitativa** | Toma categorías o etiquetas | Barrio, cantidad de ambientes |

# 2. Importar librerías

In [ ]:
# Cargamos las librerías
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Settings generales para visualizar
sns.set_theme(style="whitegrid")
pd.set_option('display.float_format', '{:.1f}'.format)

# 3. Cargar Dataset

In [ ]:
# pd.read_csv recibe la url del dataset
df = pd.read_csv('https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_jr.csv')
print("Dataset cargado exitosamente! \nDimensiones:", df.shape)

# 4. EDA, limpieza y transformaciones (básicas)

In [ ]:
# Tipos de datos y valores nulos
df.info()

Realizamos este set reducido de acciones:
* Normalizar nombres de columnas (renombrar si fuese necesario)
* Eliminar registros duplicados
* Eliminar valores nulos 
* Seleccionar variables relevantes
* Transformar dtype en los casos que sea necesario
* FE: crear columna price_usd (dividir por 1500)

In [ ]:
# Normalizar nombres de columnas:
df.columns = (
    df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)

# Renombrar neighbourhood
df = df.rename(columns={
    "neighbourhood": "barrio"
})

In [ ]:
# Eliminar registros duplicados
df.drop_duplicates(inplace=True)

In [ ]:
# Eliminar valores nulos
df.dropna(inplace=True, ignore_index=True)

In [ ]:
# Seleccionamos columnsa de interes
df.drop(columns=["host_id", "latitude", "longitude"], inplace=True)

In [ ]:
# Convertir price a float
df["price"] = df["price"].str.replace("$", "", regex=False)
df["price"] = df["price"].str.replace(",", "", regex=False)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Generar price_usd
df["price_usd"] = df["price"] / 1500

In [ ]:
# Convertir last_review a datetime
df["last_review"] = pd.to_datetime(df['last_review'], format="%Y-%m-%d", errors='coerce')

In [ ]:
df = df.sort_values(by="price_usd", ascending=True).head(650)

In [ ]:
# Corremos info() para validación final
df.info()

## 3. Histogramas

Antes de hablar de medidas de posición y de dispersión, es fundamental **visualizar** cómo se distribuyen los datos.

Un **histograma** divide el rango de valores en intervalos (bins) y muestra cuántas observaciones caen en cada uno.

- El **eje X** representa los valores de la variable
- El **eje Y** muestra la frecuencia (cantidad de observaciones)
- La **curva KDE** (Kernel Density Estimate) suaviza la distribución

In [ ]:
def plot_histograma_basic(df, variable):

    sns.histplot(
        data=df,
        x=variable,
        bins="auto",
        kde=True
    )

    plt.title(f"Distribución de {variable}")
    plt.xlabel(f"{variable}")
    plt.ylabel("Frecuencia")
    plt.xticks(rotation=90)

Playground: probar con diferentes variables:
<br>
 'barrio', 'room_type', 'bedrooms', 'price_usd', 'number_reviews', review_scores'

In [ ]:
plot_histograma_basic(df, "price_usd")

## 4. Medidas de tendencia central

Las medidas de tendencia central buscan representar un conjunto de datos con **un único valor representativo**. Las tres más usadas son:

| Medida | ¿Qué representa? | Sensible a outliers? |
|---|---|---|
| **Media** | Promedio aritmético | ✅ Sí |
| **Mediana** | Valor del medio (datos ordenados) | ❌ No |
| **Moda** | Valor más frecuente | ❌ No |

Vamos a calcularlas para la variable `precio_usd_mil`.

### 4.1 Media

La media se calcula como la suma de todos los valores dividida por la cantidad de observaciones:

$$\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}$$

In [ ]:
media = df["price_usd"].mean()
print(f"Media de precios: USD {media:.1f}")

### 4.2 Mediana

La mediana es el valor que queda en el **centro** cuando los datos están ordenados de menor a mayor.
- Si hay cantidad **impar** de datos → es el valor central
- Si hay cantidad **par** → es el promedio de los dos valores centrales

In [ ]:
mediana = df["price_usd"].median()
print(f"Mediana de precios: USD {mediana:.2f}")

In [ ]:
# Odenamos el dataframe por el campo "precio_usd_mil" y seleccionamos los registros medios
df.sort_values("price_usd").reset_index().loc[321:326,"price_usd"]


**¿Por qué la media y la mediana pueden diferir?**

Cuando hay valores extremos (outliers), la media se "corre" hacia ellos, mientras que la mediana permanece estable.
Un ejemplo clásico: si en un barrio hay un penthouse que vale 5 veces más que el resto, la media sube mucho pero la mediana apenas cambia.

### 4.3 Moda

La moda es el valor que **aparece con mayor frecuencia**. Para variables continuas como el precio es menos útil, pero tiene sentido para variables discretas como `barrio`.

In [ ]:
# Para ambientes (discreta) — más útil
moda_barrios = df["barrio"].mode()[0]
print(f"Moda de barrio: {moda_barrios}")
print()
print("Distribución de barrios:")
print(df["barrio"].value_counts().head())

Actualizamos la funcion plot_histograma para que esta versión muestre además la media y la mediana

In [ ]:
def plot_histograma_full(df, variable, desvio=False):

    sns.histplot(
        data=df,
        x=variable,
        bins="auto",
        kde=True
    )

    # Media - mediana - desvio
    media = df[variable].mean()
    mediana = df[variable].median()
    std = df[variable].std()

    plt.axvline(media, linestyle="-", color="red", label=f"Media: {media:.2f}")
    plt.axvline(mediana, linestyle="--", color="orange", label=f"Mediana: {mediana:.2f}")

    if desvio:
        plt.axvline(media + std, linestyle="--", color="green", label=f"+1std: {media + std:.2f}")
        plt.axvline(media - std, linestyle="--", color="green", label=f"-1std: {media - std:.2f}")

    plt.title(f"Distribución de {variable}")
    plt.xlabel(variable)
    plt.ylabel("Frecuencia")
    plt.xticks(rotation=90)
    plt.legend()


In [ ]:
plot_histograma_full(df, "price_usd")

## 5. Medidas de dispersión

Las medidas de tendencia central nos dicen *dónde* se concentran los datos, pero no nos dicen *cuánto varían*. Para eso existen las **medidas de dispersión**.

**Ejemplo motivador:** dos barrios pueden tener la misma media de precios pero distribuciones muy distintas. Uno puede ser muy homogéneo (todos los precios parecidos) y el otro muy heterogéneo (precios muy variados). La dispersión captura esa diferencia.

### 5.1 Rango

El rango es la diferencia entre el valor máximo y el mínimo.

In [ ]:
minimo = df["price_usd"].min()
maximo = df["price_usd"].max()
rango = maximo - minimo

print(f"Precio mínimo:  USD {minimo:.1f}")
print(f"Precio máximo:  USD {maximo:.1f}")
print(f"Rango:          USD {rango:.1f}")
print()
print("→ Un solo dato extremo puede hacer que el rango sea muy grande e informativo")

### 5.2 Varianza

La varianza mide el **promedio de las distancias al cuadrado** respecto a la media. Elevamos al cuadrado para que los desvíos positivos y negativos no se anulen.

$$s^2 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n-1}$$

> **Población vs. muestra:** cuando trabajamos con una *muestra* (que es nuestro caso), dividimos por $n-1$ en lugar de $n$. Esto corrige un sesgo estadístico y se llama **corrección de Bessel**.

In [ ]:
# Varianza de muestra (ddof=1, que es el default en pandas)
varianza = df["price_usd"].var()
print(f"Varianza (muestra): {varianza:.1f}")
print()
print("Problema: la varianza está en unidades al cuadrado (USD²), ¡difícil de interpretar!")

### 5.3 Desvío estándar

El desvío estándar es la **raíz cuadrada de la varianza**. Vuelve a las unidades originales, por lo que es mucho más interpretable.

$$s = \sqrt{s^2}$$

Una regla práctica para distribuciones aproximadamente normales: alrededor del **68%** de los datos cae dentro de ±1 desvío de la media, y el **95%** dentro de ±2 desvíos.

In [ ]:
desvio = df["price_usd"].std()
media = df["price_usd"].median()

print(f"Media:          USD {media:.2f}")
print(f"Desvío estándar: USD {desvio:.2f}")
print()
print("NOTA: si la distribución es normal se puede decir:")
print(f"Rango 1 68% de los datos (media ±1 desvío): USD {media - desvio:.2f}  →  USD {media + desvio:.2f}")
print(f"Rango 2 95% de los datos (media ±2 desvíos): USD {media - 2*desvio:.2f}  →  USD {media + 2*desvio:.2f} mil")

### Visualizamos las medidas de disperción histogramas

Analicemos primero el desvío estardar de cada barrio y luego visualicemos su distribución comparando histogramas.

In [ ]:
# Podemos usar describe de la misma manera que lo hicimos con sum() o mean() en agregaciones
df.groupby("barrio")["price_usd"].describe().sort_values(by="count", ascending=False).head(5)

In [ ]:
plt.figure(figsize=(15, 5))

# Almagro
plt.subplot(1, 2, 1)

df_palermo = df[df["barrio"] == "Palermo"]
plot_histograma_full(df_palermo, "price_usd", desvio=True)

plt.xlim(0, 200)
plt.title("Distribución de precios - Palermo")


# Belgrano
plt.subplot(1, 2, 2)

df_san_nicolas = df[df["barrio"] == "San Nicolas"]
plot_histograma_full(df_san_nicolas, "price_usd", desvio=True)

plt.xlim(0, 200)
plt.title("Distribución de precios - San Nicolas")


plt.tight_layout()
plt.show()

## 6. Cuartiles, percentiles y RIC

Los **cuartiles** dividen los datos ordenados en cuatro partes iguales:

| Cuartil | Notación | Significado |
|---|---|---|
| Primer cuartil | Q1 (percentil 25) | El 25% de los datos está por debajo |
| Segundo cuartil | Q2 (percentil 50) | = Mediana. El 50% está por debajo |
| Tercer cuartil | Q3 (percentil 75) | El 75% de los datos está por debajo |

El **Rango Intercuartil (RIC)** = Q3 - Q1, y representa la dispersión del 50% central de los datos. Es robusto frente a outliers porque ignora los extremos.

In [ ]:
Q1 = df["price_usd"].quantile(0.25)
Q2 = df["price_usd"].quantile(0.50)
Q3 = df["price_usd"].quantile(0.75)
RIC = Q3 - Q1

print(f"Q1 (percentil 25): USD {Q1:.1f}")
print(f"Q2 (percentil 50): USD {Q2:.1f}  ← mediana")
print(f"Q3 (percentil 75): USD {Q3:.1f}")
print(f"RIC (Q3 - Q1):     USD {RIC:.1f}")
print()
print("Interpretación: el 50% central de las propiedades cuesta entre")
print(f"USD {Q1:.0f} y USD {Q3:.0f}")

In [ ]:
# También podemos ver todos los percentiles con describe()
df["price_usd"].describe()

### 6.1 Detección de outliers con el criterio de Tukey

Un valor es considerado **outlier** si está por fuera de los **bigotes** del boxplot:

$$\text{Límite superior} = Q3 + 1.5 \times RIC$$
$$\text{Límite inferior} = Q1 - 1.5 \times RIC$$

Valores fuera de estos límites son inusuales y merecen atención.

In [ ]:
Q1 = df_san_nicolas["price_usd"].quantile(0.25)
Q3 = df_san_nicolas["price_usd"].quantile(0.75)
RIC = Q3 - Q1
limite_sup = Q3 + 1.5 * RIC
limite_inf = Q1 - 1.5 * RIC

print(f"Límite inferior (outlier): USD {limite_inf:.1f} mil")
print(f"Límite superior (outlier): USD {limite_sup:.1f} mil")
print()

outliers = df_san_nicolas[(df["price_usd"] < limite_inf) | (df_san_nicolas["price_usd"] > limite_sup)]
print(f"Outliers detectados: {len(outliers)}")
print()
print(outliers[["barrio", "price_usd", "bedrooms"]])

## 7. Boxplot (diagrama de caja y bigotes)

El boxplot es la visualización más completa para entender una distribución: muestra la mediana, los cuartiles, el RIC y los outliers en un solo gráfico.

**Componentes:**
- **Caja:** va de Q1 a Q3 (contiene el 50% central de los datos)
- **Línea central:** es la mediana (Q2)
- **Bigotes:** se extienden hasta el valor más extremo que **no** es outlier
- **Puntos sueltos:** outliers (valores fuera de 1.5×RIC)

Declaramos la función `plot_boxplot()` para que nos grafique un boxplot personalizado.

**El código de la función esta oculto porque no es el objetivo estudiar como se grafica, sino comprender las medidas**

In [ ]:
def plot_boxplot(df, var_x, var_y=None, order=None):
    sns.boxplot(
        data=df,
        x=var_x,
        y=var_y,
        order=order
    )

In [ ]:
barrios = ["Palermo", "San Nicolas"]
plot_boxplot(df, var_x="price_usd", var_y="barrio", order=barrios)

## Video complementario

Les sugerimos que vean este video:
[Youtube: Diagramas de Caja (BoxPlots) y Datos Anómalos (outliers) con la Regla de Tukey en Python](https://www.youtube.com/watch?v=wsfV4AO8UFw)